## Data formatting for logging files

The logging files contain irrevant/empty rows and are not formatted well (e.g., ""actor" and "object" information is mixed in the "payload" column). We will...
- Drop empty/unnecessary columns
- Remove unnecessary information from columns
- Extract the subject ("actor") and add it in the new column "actor"
- Extract the object ("context") and add it in the new column "object"

... so that the output files contains the following columns, which is the simplest format of the xAPI statement: 
- Actor (pseudo 4-character code)
- Verb (action, such as "opened", "submitted", "pressed")
- Object (the object of the action, such as "assignment", "button")
- Context (the additional information, such as "assignmentId", "pre-test", "Word explanation")
- Timestamp

Load required libraries

In [1]:
import pandas as pd
import json

Load files

In [ ]:
file_path_xapi_statement_student = r"..\raw\xapi_statement_student.xlsx"
xapi_statement_student = pd.read_excel(file_path_xapi_statement_student)

Function to drop irrelevant columns and rename columns to "verb" and "object"

In [3]:
def logging_formatting(df):
    # Drop columns 
    df = df.drop(columns=['id', 'statement_id', 'registration', 'is_voided', 'stored'])
    
    # Rename columns
    df.rename(columns={'verb_iri': 'verb', 'payload': 'object'}, inplace=True)
    
    # Remove irrelevant parts from verb column
    df['verb'] = df['verb'].str.replace('http://kibi.group/verb/', '', regex=False)

    # Define a function to clean JSON data and move the 'actor' and 'context' keys to new columns & remove unnecesary part in 'object' column
    def modify_columns_and_extract_data(json_str):
        try:
            data = json.loads(json_str, strict=False) 
        
            # Extract 'object' -> 'definition' -> 'name' -> 'en'
            object_name = None
            if isinstance(data, dict):
                obj = data.get("object", {})
                definition = obj.get("definition", {})
                name = definition.get("name", {})
                object_name = name.get("en", None)
        
            actor = data.pop('actor', None)
            context = data.pop('context', None)
            
            # Extract only the 'name' from 'actor'
            actor_name = actor.get("name") if isinstance(actor, dict) else None
        
            # Remove unnecessary keys
            keys_to_remove = ['verb', 'authority', 'stored', 'version', 'id', 'stored', 'timestamp', 'objectType']
            for key in keys_to_remove:
                data.pop(key, None)
        
            return object_name, actor_name, json.dumps(context, ensure_ascii=False) if context else None
        except (json.JSONDecodeError, TypeError):
            return json_str, None, None

    # Apply function to extract 'object' name (from "en"), 'actor', and 'context'
    df[['object', 'actor', 'context']] = df['object'].apply(lambda x: pd.Series(modify_columns_and_extract_data(x)))
    
    return df

Apply combined function to datasets

In [4]:

xapi_statement_student = logging_formatting(xapi_statement_student)

Sorting the data ascendingly and save the data to the files

In [ ]:
xapi_statement_student.sort_values("timestamp", ascending=True, inplace=True)

output_path_xapi_statement = r"..\preprocessed\xapi_statement_student_formatted.xlsx"
xapi_statement_student.to_excel(output_path_xapi_statement, index=False, engine="openpyxl")